# Capstone — Predicting Future Content Performance Decline

**Lane:** Refresh / Content Opportunity Scoring

**Research question:** Can historical search-performance signals available before a prediction date rank content items that are most likely to experience a meaningful decline in the following month?

**Decision supported:** If editorial review capacity is limited, which content items should be investigated first?

This is the final warehouse-based experiment. It uses the FlyRank full pseudonymized warehouse, constructs a genuine future-window label, compares a transparent baseline with ML, and produces a ranked review queue.

**Public-safe framing:** this is predictive ranking / decision support. It does not establish that refreshing content causes recovery and does not claim to predict Google's algorithm.

## 1. Research design

For each content item at prediction month **T0**:

```text
previous 3 monthly observations
          ↓
         T0
          ↓
next calendar month
          ↓
future decline label
```

Features are constructed only from months before T0. The target is:

> `future_decline = 1` when next-month impressions are at least 20% below the immediately preceding month's impressions.

The 20% threshold is an operational research threshold, not a universal business rule. A 10% / 20% / 30% sensitivity check is included.

The query-level 90-day snapshot is deliberately excluded from the model because its historical availability cannot be established for every prediction point without additional dated snapshots.

In [3]:
%pip -q install duckdb huggingface_hub pyarrow scikit-learn matplotlib pandas numpy

import os, getpass, json
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token: ")
if not HF_TOKEN:
    raise ValueError("A Hugging Face read token is required.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("DuckDB connected to the FlyRank warehouse.")

Note: you may need to restart the kernel to use updated packages.
DuckDB connected to the FlyRank warehouse.


In [4]:
# Inspect the release without downloading it.
print(con.sql(f"DESCRIBE SELECT * FROM {DAILY}").df()[["column_name","column_type"]].to_string(index=False))

coverage = con.sql(f'''
SELECT MIN(report_date) min_date,
       MAX(report_date) max_date,
       COUNT(DISTINCT client_hash_id) clients,
       COUNT(DISTINCT content_hash_id) content_items
FROM {DAILY}
''').df()
display(coverage)

IOException: IO Error: SSL connect error error for HTTP GET to 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/fact_content_daily_performance/month=2026-03'

LINE 1: DESCRIBE (SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse...
                               ^

## 2. Leakage-safe feature and label construction

Daily performance is first aggregated to calendar month inside DuckDB.

For each content/client/month we retain:

- monthly impressions
- monthly clicks
- impression-active days
- impression-weighted average position
- CTR

`LAG` supplies historical observations. `LEAD` supplies the following-month outcome. Rows are retained only when the lagged observations and future month are consecutive calendar months, preventing a missing month from being silently treated as a one-month transition.

In [ ]:
sql = f'''
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date)::DATE AS month,
        SUM(COALESCE(gsc_impressions,0))::DOUBLE AS impressions,
        SUM(COALESCE(gsc_clicks,0))::DOUBLE AS clicks,
        COUNT_IF(COALESCE(gsc_impressions,0) > 0) AS impression_days,
        CASE WHEN SUM(COALESCE(gsc_impressions,0)) > 0
             THEN SUM(COALESCE(gsc_avg_position,0) * COALESCE(gsc_impressions,0))
                  / SUM(COALESCE(gsc_impressions,0))
        END AS avg_position
    FROM {DAILY}
    GROUP BY 1,2,3
),
p AS (
    SELECT *,
        CASE WHEN impressions > 0 THEN clicks / impressions ELSE 0 END AS ctr,

        LAG(month,1) OVER w AS p1_month,
        LAG(month,2) OVER w AS p2_month,
        LAG(month,3) OVER w AS p3_month,

        LAG(impressions,1) OVER w AS p1_imp,
        LAG(impressions,2) OVER w AS p2_imp,
        LAG(impressions,3) OVER w AS p3_imp,

        LAG(clicks,1) OVER w AS p1_clicks,
        LAG(clicks,2) OVER w AS p2_clicks,
        LAG(clicks,3) OVER w AS p3_clicks,

        LAG(avg_position,1) OVER w AS p1_pos,
        LAG(avg_position,2) OVER w AS p2_pos,
        LAG(avg_position,3) OVER w AS p3_pos,

        LAG(impression_days,1) OVER w AS p1_days,

        LEAD(month,1) OVER w AS f1_month,
        LEAD(impressions,1) OVER w AS f1_imp
    FROM monthly
    WINDOW w AS (PARTITION BY client_hash_id,content_hash_id ORDER BY month)
)
SELECT *
FROM p
WHERE p1_imp >= 100
  AND p2_imp IS NOT NULL AND p3_imp IS NOT NULL
  AND f1_imp IS NOT NULL
  AND p1_month = month - INTERVAL 1 MONTH
  AND p2_month = p1_month - INTERVAL 1 MONTH
  AND p3_month = p2_month - INTERVAL 1 MONTH
  AND f1_month = month + INTERVAL 1 MONTH
'''
panel = con.sql(sql).df()

panel["future_decline"] = (panel["f1_imp"] < 0.80 * panel["p1_imp"]).astype(int)

eps = 1e-6
panel["imp_change_1m"] = (panel.p1_imp-panel.p2_imp)/(panel.p2_imp+eps)
panel["imp_change_3m"] = (panel.p1_imp-panel.p3_imp)/(panel.p3_imp+eps)
panel["click_change_1m"] = (panel.p1_clicks-panel.p2_clicks)/(panel.p2_clicks+eps)
panel["click_change_3m"] = (panel.p1_clicks-panel.p3_clicks)/(panel.p3_clicks+eps)
panel["position_change_1m"] = panel.p1_pos-panel.p2_pos
panel["position_change_3m"] = panel.p1_pos-panel.p3_pos
panel["p1_ctr"] = panel.p1_clicks/(panel.p1_imp+eps)
panel["p2_ctr"] = panel.p2_clicks/(panel.p2_imp+eps)
panel["p3_ctr"] = panel.p3_clicks/(panel.p3_imp+eps)
panel["ctr_change_1m"] = panel.p1_ctr-panel.p2_ctr
panel["log_p1_imp"] = np.log1p(panel.p1_imp)
panel["log_p1_clicks"] = np.log1p(panel.p1_clicks)

FEATURES = [
    "p1_imp","p2_imp","p3_imp","p1_clicks","p2_clicks","p3_clicks",
    "p1_pos","p2_pos","p3_pos","p1_days","p1_ctr","p2_ctr","p3_ctr",
    "imp_change_1m","imp_change_3m","click_change_1m","click_change_3m",
    "position_change_1m","position_change_3m","ctr_change_1m",
    "log_p1_imp","log_p1_clicks"
]

data = panel.dropna(subset=FEATURES).copy()
print(f"Eligible prediction rows: {len(data):,}")
print(f"Clients: {data.client_hash_id.nunique():,}")
print(f"Content items: {data.content_hash_id.nunique():,}")
print(f"Prediction months: {data.month.min()} → {data.month.max()}")
print(f"Future-decline base rate: {data.future_decline.mean():.1%}")

## 3. Validation

The final evaluation is chronological: the latest 20% of eligible prediction months are held out as test data.

This matches the production question: learn from earlier periods, then rank a later period. A random row split is intentionally not used.

The test base rate must be reported beside Precision@K because precision can look high when the positive class is common.

In [ ]:
months = np.sort(data.month.unique())
if len(months) < 6:
    raise ValueError(f"Only {len(months)} prediction months available; at least 6 are required.")

n_test = max(2, int(np.ceil(len(months)*0.20)))
test_months = months[-n_test:]
train_months = months[:-n_test]

train = data[data.month.isin(train_months)].copy()
test = data[data.month.isin(test_months)].copy()

print("TRAIN:", train_months[0], "→", train_months[-1], f"({len(train):,} rows)")
print("TEST :", test_months[0], "→", test_months[-1], f"({len(test):,} rows)")
print(f"Train base rate: {train.future_decline.mean():.1%}")
print(f"Test base rate:  {test.future_decline.mean():.1%}")

## 4. Baseline

The baseline is a transparent historical-deterioration score:

- 50% weight: negative impression movement
- 30% weight: negative click movement
- 20% weight: worsening position

It uses only pre-T0 information and is evaluated on exactly the same temporal test set as the ML models.

In [ ]:
def precision_at_k(y, score, k):
    t = pd.DataFrame({"y": np.asarray(y), "score": np.asarray(score)})
    return float(t.sort_values("score", ascending=False).head(min(k,len(t))).y.mean())

def baseline_score(x):
    return (
        0.50 * (-x.imp_change_1m.clip(-5,5)) +
        0.30 * (-x.click_change_1m.clip(-5,5)) +
        0.20 * (x.position_change_1m.clip(-20,20)/10)
    )

y_test = test.future_decline.astype(int)
base_score = baseline_score(test)

baseline_metrics = {
    f"Precision@{k}": precision_at_k(y_test, base_score, k)
    for k in [10,25,50,100]
}
baseline_metrics

## 5. Models

We compare:

1. Logistic Regression — simple linear benchmark.
2. Random Forest — nonlinear tabular model.

Model selection is based on Precision@50, matching the editorial top-of-queue decision.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score

X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train.future_decline.astype(int), test.future_decline.astype(int)

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=50,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42
    )
}

rows, probs = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:,1]
    probs[name] = p
    rows.append({
        "model": name,
        "ROC_AUC": roc_auc_score(y_test,p),
        "PR_AUC": average_precision_score(y_test,p),
        "Precision@10": precision_at_k(y_test,p,10),
        "Precision@25": precision_at_k(y_test,p,25),
        "Precision@50": precision_at_k(y_test,p,50),
        "Precision@100": precision_at_k(y_test,p,100),
        "Recall@0.5": recall_score(y_test,p>=.5,zero_division=0),
        "F1@0.5": f1_score(y_test,p>=.5,zero_division=0)
    })

rows.append({
    "model":"historical_rule_baseline",
    "ROC_AUC":roc_auc_score(y_test,base_score),
    "PR_AUC":average_precision_score(y_test,base_score),
    **baseline_metrics,
    "Recall@0.5":np.nan,
    "F1@0.5":np.nan
})

results = pd.DataFrame(rows).sort_values(["Precision@50","PR_AUC"],ascending=False)
display(results.style.format({c:"{:.3f}" for c in results.columns if c!="model"}))

best_name = results[results.model!="historical_rule_baseline"].iloc[0].model
best_p = probs[best_name]
print("Selected ML model:", best_name)

In [ ]:
# Main result chart.
ks = [10,25,50,100]
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(ks,[baseline_metrics[f"Precision@{k}"] for k in ks],marker="o",label="Baseline")
ax.plot(ks,[precision_at_k(y_test,best_p,k) for k in ks],marker="o",label=best_name)
ax.set_xlabel("K"); ax.set_ylabel("Precision@K")
ax.set_title("Future-decline ranking performance")
ax.grid(alpha=.25); ax.legend()
plt.tight_layout()
plt.show()

## 6. Feature interpretation

Feature importance is descriptive, not causal. It tells us which historical signals the fitted model used most, not which signals cause future decline.

In [ ]:
if best_name == "random_forest":
    rf = models["random_forest"]
    importance = pd.DataFrame({
        "feature": FEATURES,
        "importance": rf.feature_importances_
    }).sort_values("importance",ascending=False).head(15)
    display(importance)

    fig, ax = plt.subplots(figsize=(8,6))
    ax.barh(importance.feature[::-1],importance.importance[::-1])
    ax.set_title("Top model features")
    ax.set_xlabel("Random Forest importance")
    plt.tight_layout()
    plt.show()

In [ ]:
# Error analysis.
err = test[["client_hash_id","content_hash_id","month","future_decline",
            "p1_imp","p1_clicks","p1_pos","imp_change_1m",
            "click_change_1m","position_change_1m"]].copy()
err["score"] = best_p
err["predicted_decline"] = (best_p >= .5).astype(int)
err["error_type"] = np.select([
    (err.predicted_decline==1)&(err.future_decline==0),
    (err.predicted_decline==0)&(err.future_decline==1)
],["false_positive","false_negative"],default="correct")

print(err.error_type.value_counts())
display(err[err.error_type=="false_positive"].sort_values("score",ascending=False).head(10))
display(err[err.error_type=="false_negative"].sort_values("score").head(10))

## 7. Threshold sensitivity

The 20% target is an operational choice. We test 10%, 20%, and 30% future-impression decline thresholds on the same temporal test rows.

In [ ]:
sens = []
for d in [.10,.20,.30]:
    yy = (test.f1_imp < (1-d)*test.p1_imp).astype(int)
    sens.append({
        "decline_threshold":d,
        "test_base_rate":yy.mean(),
        "Precision@50":precision_at_k(yy,best_p,50),
        "Precision@100":precision_at_k(yy,best_p,100)
    })
sensitivity = pd.DataFrame(sens)
display(sensitivity)

## 8. Ranked recommendations

The final operational queue is generated after evaluation by fitting the selected model on all eligible historical rows and scoring the latest available prediction month.

Actions are decision-support labels:

- **refresh_review:** highest-risk items with meaningful historical visibility
- **priority_review:** moderate/high risk requiring manual investigation
- **monitor:** lower-risk items

These labels do not imply that the action will cause recovery.

In [ ]:
final_model = models[best_name]
final_model.fit(data[FEATURES],data.future_decline.astype(int))

latest_month = data.month.max()
latest = data[data.month==latest_month].copy()
latest["risk_score"] = final_model.predict_proba(latest[FEATURES])[:,1]

latest["reason_code"] = np.select([
    (latest.risk_score>=.75)&(latest.imp_change_1m<0),
    (latest.risk_score>=.50)&(latest.position_change_1m>0),
    latest.risk_score>=.50
],[
    "high_risk_with_visibility_deterioration",
    "priority_review_position_deterioration",
    "priority_review"
],default="monitor")

latest["action"] = np.select([
    latest.risk_score>=.75,
    latest.risk_score>=.50
],["refresh_review","priority_review"],default="monitor")

queue = latest[[
    "client_hash_id","content_hash_id","month","risk_score","action","reason_code",
    "p1_imp","p1_clicks","p1_pos","imp_change_1m","click_change_1m","position_change_1m"
]].sort_values("risk_score",ascending=False).reset_index(drop=True)
queue.insert(0,"rank",np.arange(1,len(queue)+1))

display(queue.head(25))

## 9. Public-safe interpretation

**Observed:** the model's measured ranking performance on the held-out future period.

**Directional:** feature importance and error patterns indicate which historical signal families were useful to this model.

**Decision-support:** the ranked queue identifies items for human review.

**Not claimed:** causal refresh impact, Google's ranking algorithm, guaranteed traffic recovery, or universal thresholds.

In [ ]:
# Reproducibility receipts. Do not write the raw warehouse to disk.
out = Path("../outputs")
if not out.exists(): out = Path("work/outputs")
out.mkdir(parents=True,exist_ok=True)

results.to_csv(out/"future_model_results.csv",index=False)
sensitivity.to_csv(out/"future_threshold_sensitivity.csv",index=False)
queue.to_csv(out/"future_ranked_recommendations.csv",index=False)

payload = {
    "task":"future_content_performance_decline_ranking",
    "label":"next-month impressions < 80% of previous-month impressions",
    "feature_window":"previous three consecutive monthly observations",
    "test_months":[str(x) for x in test_months],
    "random_seed":42,
    "test_base_rate":float(y_test.mean()),
    "best_model":best_name,
    "metrics":results.to_dict(orient="records"),
    "threshold_sensitivity":sensitivity.to_dict(orient="records")
}
(out/"future_metrics.json").write_text(json.dumps(payload,indent=2),encoding="utf-8")

print("Artifacts written to:",out.resolve())

## 10. Five-minute demo

**0:00–0:45:** Research question and editorial decision.

**0:45–1:30:** Full warehouse → DuckDB → monthly panel.

**1:30–2:15:** Leakage-safe temporal design: previous 3 months as features, next month as label.

**2:15–3:15:** Baseline vs ML Precision@K and test base rate.

**3:15–4:15:** Feature importance and error analysis.

**4:15–5:00:** Ranked queue, limitations, and decision-support framing.

## Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset**. Data credit: https://flyrank.ai

No client names, domains, URLs, private queries, credentials, or raw exports are published.

## Final self-check

- [ ] Full warehouse queried through DuckDB; warehouse not committed.
- [ ] Features use only periods before the prediction point.
- [ ] Future month is used only for the label.
- [ ] `trend_direction` / `trend_pct` are not used.
- [ ] Pseudonymous IDs are not predictive features.
- [ ] Baseline and ML use the same temporal test set.
- [ ] Test base rate is reported next to Precision@K.
- [ ] Metrics JSON is committed as the reproducibility receipt.
- [ ] Error analysis and threshold sensitivity are included.
- [ ] Public outputs contain no client-identifying details.
- [ ] Paper claims use observed/measured/directional/decision-support language.
- [ ] `submission/paper_url.txt` is filled only after the paper is deployed.